# AgroSense: YOLO26s-cls train + export (lean version)
Kaggle Settings > Accelerator > **GPU** (e.g., P100 or T4x2). Run cells top to bottom. No evaluation code: train, save, export only.[cite: 1]

In [ ]:
# 1. Install
!pip -q install -U ultralytics onnx onnxslim onnxruntime

In [ ]:
# 2. Unzip uploaded data to working directory
# Assuming 'agrosense_clean.zip' has been uploaded directly to /kaggle/working/
!rm -rf /kaggle/working/unified_clean
!unzip -q /kaggle/working/agrosense_clean.zip -d /kaggle/working/unified_clean

In [ ]:
# 3. Balance the TRAIN set only (val untouched)
import random, shutil
from pathlib import Path
random.seed(42)

SRC, DST = Path("/kaggle/working/unified_clean"), Path("/kaggle/working/balanced")
MAX_TRAIN = 1000
MIN_TRAIN = 300

shutil.rmtree(DST, ignore_errors=True)
shutil.copytree(SRC / "val", DST / "val")
for cdir in sorted((SRC / "train").iterdir()):
    files = sorted(cdir.glob("*.jpg")); random.shuffle(files); files = files[:MAX_TRAIN]
    out = DST / "train" / cdir.name; out.mkdir(parents=True)
    for f in files:
        shutil.copy2(f, out / f.name)
    for j in range(max(0, MIN_TRAIN - len(files))):
        f = files[j % len(files)]
        shutil.copy2(f, out / f"dup{j}_{f.name}")
    print(f"{cdir.name:28s} train={len(list(out.glob('*.jpg'))):5d}")
shutil.rmtree(SRC)   # free disk space

In [ ]:
# 4. Train (memory-friendly settings)
from ultralytics import YOLO
IMGSZ = 320

model = YOLO("yolo26s-cls.pt")
model.train(
    data="/kaggle/working/balanced", imgsz=IMGSZ, epochs=60, patience=10,
    batch=32, workers=2, cache=False, amp=True,      # lower batch/workers = less memory
    dropout=0.2, label_smoothing=0.1,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, degrees=20, scale=0.5, fliplr=0.5, flipud=0.2, erasing=0.4,
    project="/kaggle/working/runs", name="agrosense", exist_ok=True, plots=False,
)

In [ ]:
# 5. Verify weights in working directory
# Kaggle automatically persists /kaggle/working/ files to the Output tab after saving the notebook.
!ls -lh /kaggle/working/runs/agrosense/weights

In [ ]:
# 6. Free memory before export
import gc, torch
del model
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 7. Export .pt / .onnx / .tflite (fp16) + labels.json
import json, glob, shutil
from ultralytics import YOLO
BEST = "/kaggle/working/runs/agrosense/weights/best.pt"
OUT = Path("/kaggle/working/agrosense_release"); shutil.rmtree(OUT, ignore_errors=True); OUT.mkdir()

m = YOLO(BEST)
shutil.copy(BEST, OUT / "agrosense_cls.pt")

shutil.copy(m.export(format="onnx", imgsz=IMGSZ, simplify=True, device="cpu"), OUT / "agrosense_cls.onnx")

m.export(format="tflite", imgsz=IMGSZ, half=True, device="cpu")
cands = glob.glob("/kaggle/working/runs/agrosense/weights/**/*float16.tflite", recursive=True) \
     or glob.glob("/kaggle/working/runs/agrosense/weights/**/*.tflite", recursive=True)
print("tflite found:", cands)
shutil.copy(cands[0], OUT / "agrosense_cls_fp16.tflite")

labels = [m.names[i] for i in sorted(m.names)]
json.dump({"imgsz": IMGSZ, "labels": labels}, open(OUT / "labels.json", "w"), indent=2)
!ls -lh /kaggle/working/agrosense_release

In [ ]:
# 8. Package release files for Kaggle Output Download
!cd /kaggle/working/ && zip -r agrosense_release.zip agrosense_release/
print("Done: Check the Kaggle 'Output' pane to download agrosense_release.zip (.pt, .onnx, .tflite, labels.json)")